In [2]:
#cell 1

import os
import gc
import json
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate
# 📦 Imports
import json
from collections import Counter
from pathlib import Path
import os
import evaluate
import pandas as pd
from datasets import Dataset, DatasetDict


# 📊 Evaluation metric
from evaluate import load
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments, EarlyStoppingCallback
)

accuracy_metric = load("accuracy")


/data/literature/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/data/literature/.venv/lib/python3.12/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
#cell 2
# Set this before importing any other libraries that use CUDA
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

# Clear CUDA cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

print("✅ Using only GPU 3")
print("🧹 Cleared CUDA cache")

# Verify single GPU setup
if torch.cuda.is_available():
    print(f"Available GPUs: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f" - GPU {i}: {torch.cuda.get_device_name(i)} (Physical GPU 3)")
else:
    print("❌ No CUDA available!")
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1" 

✅ Using only GPU 3
🧹 Cleared CUDA cache
Available GPUs: 1
 - GPU 0: NVIDIA RTX A6000 (Physical GPU 3)


In [9]:
import os, json
import pandas as pd

# Base scraper project path
SCRAPER_PROJECT_PATH = "/home/literature/hindwi-scraper"
POETS_ROOT_DIR = os.path.join(SCRAPER_PROJECT_PATH, "output", "poets")

def load_hindi_poems(scraper_base_path):
    """
    Loads Hindi poems and their categories using metadata.json and devnagri_path.
    Each row = one poem, categories stored as list.
    """
    dataset = []

    print(f"🔍 Scanning poets in: {POETS_ROOT_DIR}")
    if not os.path.isdir(POETS_ROOT_DIR):
        print(f"❌ Error: Directory '{POETS_ROOT_DIR}' not found.")
        return pd.DataFrame([])

    for poet_name in os.listdir(POETS_ROOT_DIR):
        poet_dir_path = os.path.join(POETS_ROOT_DIR, poet_name)
        if not os.path.isdir(poet_dir_path):
            continue

        metadata_file = os.path.join(poet_dir_path, "metadata.json")
        if not os.path.exists(metadata_file):
            continue

        try:
            with open(metadata_file, 'r', encoding='utf-8') as f:
                metadata = json.load(f)

            for poem_info in metadata.get("kavita", []):
                categories = poem_info.get("categories")
                path_from_json = poem_info.get("devnagri_path")

                # ✅ Only consider Devanagari poems with categories
                if categories and path_from_json:
                    absolute_path = os.path.join(scraper_base_path, path_from_json)
                    if os.path.exists(absolute_path):
                        with open(absolute_path, 'r', encoding='utf-8') as poem_file:
                            poem_text = poem_file.read()

                        dataset.append({
                            "text": poem_text.strip(),
                            "categories": categories,
                            "poet": poet_name
                        })
                    else:
                        print(f"⚠️ Missing file at: {absolute_path}")

        except Exception as e:
            print(f"❌ Error processing {metadata_file}: {e}")

    return pd.DataFrame(dataset)

# Load dataset
df = load_hindi_poems(SCRAPER_PROJECT_PATH)

print(f"✅ Loaded {len(df)} Hindi poems (Devanagari script only)")
print(df.head())


🔍 Scanning poets in: /home/literature/hindwi-scraper/output/poets
✅ Loaded 20314 Hindi poems (Devanagari script only)
                                                text             categories  \
0  वर्षा की सारी शाम\nमेरी आँखें बरामदे में बैठी ...         [असमिया कविता]   
1  यह सामने खड़ा पदार्थ\nमेरी शब्द-घास को चबा जाएग...  [आत्म, गुजराती कविता]   
2  स्थिर, अधखुली खिड़की\nपीपल के पत्ते पीले\nधूप म...  [आत्म, गुजराती कविता]   
3  हम जब जंगल से निकल कर\nबस्ती में आए\nतो अपने न...        [दरवाज़ा, लौटना]   
4  नानी हर रात\nइक वही कहानी सुनाती थी\nबच्चे सुन...        [प्रेम, स्मृति]   

                 poet  
0          mahim-bora  
1  labhshankar-thakar  
2  labhshankar-thakar  
3         vineet-raja  
4         vineet-raja  


In [11]:

# 🔹 Explode categories for counting
import json
df_exploded = df.explode("categories")

# 🔹 Count category frequencies
category_counts = df_exploded["categories"].value_counts()

print("📊 Category distribution (before filtering):")
print(category_counts.head(20))  # show top 20

# Save category counts to JSON
category_counts.to_json("category_counts.json", orient="index", force_ascii=False)

print("✅ Category counts saved to category_counts.json")




📊 Category distribution (before filtering):
categories
स्त्री           1527
प्रेम            1415
चीज़ें            1045
वैश्विक कविता    1035
जीवन             1035
संबंध             874
लोक               767
समय               760
आत्म              745
स्मृति            738
कविता             605
मृत्यु            559
प्रतिरोध          524
संघर्ष            511
यात्रा            498
प्रकृति           491
निंदा             466
दुख               464
हिंसा             456
पंजाबी कविता      447
Name: count, dtype: int64
✅ Category counts saved to category_counts.json
